In [1]:
#Author: Eve Watson
#This is the python file for theLook ecommerce analytics project
#Import neccessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import missingno as ms

In [3]:
#Load in data and make them into tables
orders = pd.read_csv('/content/drive/MyDrive/thelook/Look_Orders.csv')
order_items = pd.read_csv('/content/drive/MyDrive/thelook/Look_order_items.csv')
products = pd.read_csv('/content/drive/MyDrive/thelook/Look_products.csv')
events = pd.read_csv('/content/drive/MyDrive/thelook/Look_events.csv')
users = pd.read_csv('/content/drive/MyDrive/thelook/Look_users.csv')

tables = {
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "events": events,
    "users": users
}

/tmp/ipykernel_2708/1205535394.py:6: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  users = pd.read_csv('/content/drive/MyDrive/thelook/Look_users.csv')


In [4]:
#Got a warning about data types in the user column, quickly inspecting it before data validation
print(users.columns[8])
print(users.iloc[:, 8].head(20))
print(users.iloc[:, 8].map(type).value_counts())

postal_code
0     69980-000
1     69980-000
2     69980-000
3     69980-000
4     69980-000
5     69980-000
6     69980-000
7     69980-000
8     69980-000
9     69980-000
10    69980-000
11    69980-000
12    69980-000
13    69980-000
14    69980-000
15    69980-000
16    69980-000
17    69970-000
18    69970-000
19    69970-000
Name: postal_code, dtype: object
postal_code
<class 'str'>    98304
<class 'int'>     1696
Name: count, dtype: int64


In [5]:
#Postal code has mixed data types, and its giving the system some grief, So standardize it to a string class since there should be no aggregate calculations beyond needing to cound them
users['postal_code'] = (
    users['postal_code']
    .astype(str)
    .str.replace('.0', '', regex=False)
)
print(users['postal_code'].value_counts().head(20))
print(users['postal_code'].map(type).value_counts())

postal_code
02675-031    412
70297-400    350
40301-110    249
40301-155    203
361006       133
82590-300    106
81020-490     95
59            94
25265-008     92
65137-000     78
518102        78
68515-000     77
66            76
523900        76
523850        75
750000        73
518024        72
430032        70
116100        70
518109        69
Name: count, dtype: int64
postal_code
<class 'str'>    100000
Name: count, dtype: int64


In [12]:
#viewing data table by table
for table_name, i in tables.items():
    print(f"\n TABLE: {table_name.upper()}")
    print(f"{'-'*20}")

    #Number of rows and columns
    print("\nShape (rows, columns):")
    print(i.shape)

    #First 5 rows
    print("\nFirst 5 rows:")
    display(i.head())

    #Column names, data types, and non-null counts
    print("\nTable information:")
    i.info()


 TABLE: ORDERS
--------------------

Shape (rows, columns):
(124452, 9)

First 5 rows:


,order_id,user_id,status,gender,created_at,returned_at,shipped_at,delivered_at,num_of_item
0,6,4,Cancelled,F,2026-03-28 20:13:14 UTC,NaN,NaN,NaN,2
1,8,5,Cancelled,F,2024-08-13 18:25:40 UTC,NaN,NaN,NaN,3
2,38,28,Cancelled,F,2026-02-03 03:46:46 UTC,NaN,NaN,NaN,2
3,52,37,Cancelled,F,2024-07-08 20:05:34 UTC,NaN,NaN,NaN,1
4,55,41,Cancelled,F,2026-09-02 11:34:36 UTC,NaN,NaN,NaN,1



Table information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124452 entries, 0 to 124451
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   order_id      124452 non-null  int64 
 1   user_id       124452 non-null  int64 
 2   status        124452 non-null  object
 3   gender        124452 non-null  object
 4   created_at    124452 non-null  object
 5   returned_at   12396 non-null   object
 6   shipped_at    81063 non-null   object
 7   delivered_at  43755 non-null   object
 8   num_of_item   124452 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 8.5+ MB

 TABLE: ORDER_ITEMS
--------------------

Shape (rows, columns):
(180746, 11)

First 5 rows:


,id,order_id,user_id,product_id,inventory_item_id,status,created_at,shipped_at,delivered_at,returned_at,sale_price
0,3671,2526,1976,14235,9917,Complete,2026-04-11 23:26:05 UTC,2026-04-13 05:58:38 UTC,2026-04-15 09:41:38 UTC,NaN,0.02
1,26255,18153,14471,14235,70805,Processing,2024-09-08 20:56:12 UTC,NaN,NaN,NaN,0.02
2,69716,47981,38538,14235,188113,Processing,2026-06-23 12:07:16 UTC,NaN,NaN,NaN,0.02
3,122351,84256,67645,14235,330343,Processing,2026-10-25 03:50:47.130257 UTC,NaN,NaN,NaN,0.02
4,503,348,268,14235,1375,Shipped,2020-11-18 00:45:23 UTC,2020-11-19 16:12:46 UTC,NaN,NaN,0.02



Table information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180746 entries, 0 to 180745
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 180746 non-null  int64  
 1   order_id           180746 non-null  int64  
 2   user_id            180746 non-null  int64  
 3   product_id         180746 non-null  int64  
 4   inventory_item_id  180746 non-null  int64  
 5   status             180746 non-null  object 
 6   created_at         180746 non-null  object 
 7   shipped_at         117847 non-null  object 
 8   delivered_at       63571 non-null   object 
 9   returned_at        18039 non-null   object 
 10  sale_price         180746 non-null  float64
dtypes: float64(1), int64(5), object(5)
memory usage: 15.2+ MB

 TABLE: PRODUCTS
--------------------

Shape (rows, columns):
(29120, 9)

First 5 rows:


,id,cost,category,name,brand,retail_price,department,sku,distribution_center_id
0,13842,2.51875,Accessories,Low Profile Dyed Cotton Twill Cap - Navy W39S55D,MG,6.25,Women,EBD58B8A3F1D72F4206201DA62FB1204,1
1,13928,2.33835,Accessories,Low Profile Dyed Cotton Twill Cap - Putty W39S55D,MG,5.95,Women,2EAC42424D12436BDD6A5B8A88480CC3,1
2,14115,4.87956,Accessories,Enzyme Regular Solid Army Caps-Black W35S45D,MG,10.99,Women,EE364229B2791D1EF9355708EFF0BA34,1
3,14157,4.64877,Accessories,Enzyme Regular Solid Army Caps-Olive W35S45D (...,MG,10.99,Women,00BD13095D06C20B11A2993CA419D16B,1
4,14273,6.50793,Accessories,Washed Canvas Ivy Cap - Black W11S64C,MG,15.99,Women,F531DC20FDE20B7ADF3A73F52B71D0AF,1



Table information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29120 entries, 0 to 29119
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      29120 non-null  int64  
 1   cost                    29120 non-null  float64
 2   category                29120 non-null  object 
 3   name                    29118 non-null  object 
 4   brand                   29096 non-null  object 
 5   retail_price            29120 non-null  float64
 6   department              29120 non-null  object 
 7   sku                     29120 non-null  object 
 8   distribution_center_id  29120 non-null  int64  
dtypes: float64(2), int64(2), object(5)
memory usage: 2.0+ MB

 TABLE: EVENTS
--------------------

Shape (rows, columns):
(2421707, 13)

First 5 rows:


,id,user_id,sequence_number,session_id,created_at,ip_address,city,state,postal_code,browser,traffic_source,uri,event_type
0,2187295,NaN,3,26595b69-d111-42fe-a09a-06adfc68008d,2020-06-17 13:29:00 UTC,105.163.208.109,São Paulo,São Paulo,02675-031,Safari,Adwords,/cancel,cancel
1,2011078,NaN,3,b5e386ef-38c2-4882-873d-84e94455a932,2022-04-28 06:36:00 UTC,107.193.140.223,São Paulo,São Paulo,02675-031,Chrome,Adwords,/cancel,cancel
2,2126069,NaN,3,5f2bea6a-c2a2-4d83-a642-f25f2a50d42d,2020-01-07 18:52:00 UTC,180.71.34.190,São Paulo,São Paulo,02675-031,IE,Email,/cancel,cancel
3,1323778,NaN,3,ca5ce5c2-c47b-47e9-a522-a18bde56d97d,2025-08-07 07:40:00 UTC,114.150.77.95,São Paulo,São Paulo,02675-031,Chrome,YouTube,/cancel,cancel
4,1401590,NaN,3,e033f232-bf98-410f-8c1a-84e10b7c6338,2021-06-03 13:09:00 UTC,202.223.17.133,São Paulo,São Paulo,02675-031,Firefox,Facebook,/cancel,cancel



Table information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2421707 entries, 0 to 2421706
Data columns (total 13 columns):
 #   Column           Dtype  
---  ------           -----  
 0   id               int64  
 1   user_id          float64
 2   sequence_number  int64  
 3   session_id       object 
 4   created_at       object 
 5   ip_address       object 
 6   city             object 
 7   state            object 
 8   postal_code      object 
 9   browser          object 
 10  traffic_source   object 
 11  uri              object 
 12  event_type       object 
dtypes: float64(1), int64(2), object(10)
memory usage: 240.2+ MB

 TABLE: USERS
--------------------

Shape (rows, columns):
(100000, 16)

First 5 rows:


,id,first_name,last_name,email,age,gender,state,street_address,postal_code,city,country,latitude,longitude,traffic_source,created_at,user_geom
0,81146,Brittany,Newton,brittanynewton@example.net,62,F,Acre,6040 Webb Mountains,69980-000,NaN,Brasil,-8.065346,-72.870949,Search,2025-05-05 15:58:00 UTC,POINT(-72.87094866 -8.065346116)
1,56556,Kim,Castillo,kimcastillo@example.net,36,F,Acre,6637 Roberson Meadows Apt. 548,69980-000,NaN,Brasil,-8.065346,-72.870949,Search,2026-01-30 03:19:00 UTC,POINT(-72.87094866 -8.065346116)
2,2792,Debbie,Smith,debbiesmith@example.net,27,F,Acre,139 Wheeler Pass Suite 607,69980-000,NaN,Brasil,-8.065346,-72.870949,Search,2026-10-27 06:00:53.805757 UTC,POINT(-72.87094866 -8.065346116)
3,18073,Robert,Watkins,robertwatkins@example.com,17,M,Acre,32979 Leach Falls,69980-000,NaN,Brasil,-8.065346,-72.870949,Search,2022-10-31 12:32:00 UTC,POINT(-72.87094866 -8.065346116)
4,35509,Shirley,Cox,shirleycox@example.net,41,F,Acre,14688 Jonathan Rest,69980-000,NaN,Brasil,-8.065346,-72.870949,Search,2024-02-27 17:48:00 UTC,POINT(-72.87094866 -8.065346116)



Table information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 16 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              100000 non-null  int64  
 1   first_name      100000 non-null  object 
 2   last_name       100000 non-null  object 
 3   email           100000 non-null  object 
 4   age             100000 non-null  int64  
 5   gender          100000 non-null  object 
 6   state           100000 non-null  object 
 7   street_address  100000 non-null  object 
 8   postal_code     100000 non-null  object 
 9   city            99001 non-null   object 
 10  country         100000 non-null  object 
 11  latitude        100000 non-null  float64
 12  longitude       100000 non-null  float64
 13  traffic_source  100000 non-null  object 
 14  created_at      100000 non-null  object 
 15  user_geom       100000 non-null  object 
dtypes: float64(2), int64(2), object(12)
m

In [13]:
#Checking for null counts
for table_name, i in tables.items():
    print(f"\n{table_name.upper()} — Missing Values")
    print(i.isnull().sum())


ORDERS — Missing Values
order_id             0
user_id              0
status               0
gender               0
created_at           0
returned_at     112056
shipped_at       43389
delivered_at     80697
num_of_item          0
dtype: int64

ORDER_ITEMS — Missing Values
id                        0
order_id                  0
user_id                   0
product_id                0
inventory_item_id         0
status                    0
created_at                0
shipped_at            62899
delivered_at         117175
returned_at          162707
sale_price                0
dtype: int64

PRODUCTS — Missing Values
id                         0
cost                       0
category                   0
name                       2
brand                     24
retail_price               0
department                 0
sku                        0
distribution_center_id     0
dtype: int64

EVENTS — Missing Values
id                       0
user_id            1123678
sequence_number         

In [14]:
#Primary key duplicate checks
primary_keys = {
    "orders": "order_id",
    "order_items": "id",
    "products": "id",
    "events": "id",
    "users": "id"
}

for table_name, i in tables.items():
    keys = primary_keys[table_name]

    print(f"\n- {table_name.upper()} -")

    #Check for duplicate primary keys
    duplicates = i[i[keys].duplicated(keep=False)]

    print(f"Duplicate {keys} count: {i[keys].duplicated().sum()}")
    display(duplicates.sort_values(by=keys) if not duplicates.empty else duplicates)


- ORDERS -
Duplicate order_id count: 0


,order_id,user_id,status,gender,created_at,returned_at,shipped_at,delivered_at,num_of_item



- ORDER_ITEMS -
Duplicate id count: 0


,id,order_id,user_id,product_id,inventory_item_id,status,created_at,shipped_at,delivered_at,returned_at,sale_price



- PRODUCTS -
Duplicate id count: 0


,id,cost,category,name,brand,retail_price,department,sku,distribution_center_id



- EVENTS -
Duplicate id count: 0


,id,user_id,sequence_number,session_id,created_at,ip_address,city,state,postal_code,browser,traffic_source,uri,event_type



- USERS -
Duplicate id count: 0


,id,first_name,last_name,email,age,gender,state,street_address,postal_code,city,country,latitude,longitude,traffic_source,created_at,user_geom


In [15]:
#Value range for sales and product data before IQR testing
print("price range\n--",order_items["sale_price"].describe().to_string(), sep =" ") #Dist of price range for items
print("Production costs\n--",products["cost"].describe().to_string()) #Dist of production cost
print("Retail Prices\n--",products["retail_price"].describe().to_string()) #Dist of retail price
print("User Ages\n--",users["age"].describe().to_string()) #Dist of user age
print("Number of Orders\n--",orders["num_of_item"].describe().to_string()) #Dist of order quantity

price range
-- count    180746.000000
mean         59.583309
std          66.444946
min           0.020000
25%          24.700001
50%          39.990002
75%          69.949997
max         999.000000
Production costs
-- count    29120.000000
mean        28.481774
std         30.624681
min          0.008300
25%         11.275613
50%         19.675081
75%         34.440000
max        557.151002
Retail Prices
-- count    29120.000000
mean        59.220164
std         65.888927
min          0.020000
25%         24.000000
50%         39.990002
75%         69.949997
max        999.000000
User Ages
-- count    100000.000000
mean         41.028310
std          17.049826
min          12.000000
25%          26.000000
50%          41.000000
75%          56.000000
max          70.000000
Number of Orders
-- count    124452.000000
mean          1.452335
std           0.806450
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max           4.000000


In [16]:
#Now IQR
Q1= order_items["sale_price"].quantile(0.25)
Q3= order_items["sale_price"].quantile(0.75)
IQR= Q3 - Q1
lower= (Q1 - 1.5) * IQR
upper= (Q3 + 1.5) * IQR
outliers= order_items[
    (order_items["sale_price"] < lower) | (order_items["sale_price"] > upper)
]
print("Sale Price Outliers:", len(outliers))
print(f"IQR range: ${lower:.2f}  ${upper:.2f}")

Sale Price Outliers: 180746
IQR range: $1049.80  $3233.11


In [17]:
#Now for the catagorical data distributions
print("Order Status:\n--",orders["status"].value_counts().to_string(),"\n")
print("Genders:\n--",users["gender"].value_counts().to_string(),"\n")
print("Users per country:\n--",users["country"].value_counts().head(10).to_string(),"\n")
print("Traffic Source:\n--",users["traffic_source"].value_counts().to_string(),"\n")
print("Product Categories\n--",products["category"].value_counts().head(10).to_string(),"\n")
print("Product Catalogue:\n--",products["department"].value_counts().to_string(),"\n")
print("Order Statuses:\n--",order_items["status"].value_counts().to_string(),"\n")

Order Status:
-- status
Shipped       37308
Complete      31359
Processing    24755
Cancelled     18634
Returned      12396 

Genders:
-- gender
M    50195
F    49805 

Users per country:
-- country
China             33738
United States     22750
Brasil            14576
South Korea        5384
France             4701
United Kingdom     4510
Germany            4198
Spain              4021
Japan              2438
Australia          2231 

Traffic Source:
-- traffic_source
Search      70294
Organic     14692
Facebook     6034
Email        5021
Display      3959 

Product Categories
-- category
Intimates                        2363
Jeans                            1999
Tops & Tees                      1868
Fashion Hoodies & Sweatshirts    1866
Swim                             1798
Sleep & Lounge                   1771
Shorts                           1765
Sweaters                         1737
Accessories                      1559
Active                           1432 

Product Catalogue:
-

In [18]:
#Data Cleaning for certain missing values

#From the data validation, there were null brand names, change that to "unknown"
null_brands = products["brand"].isna() | (products["brand"] == "")
products.loc[null_brands, "brand"] = "Unknown"

#There were also two null product names, fix that as well
null_names = products["name"].isna() | (products["name"] == "")
products.loc[null_names, "name"] = "Unknown Product"

#Also fill in empty user Cities
null_cities = users["city"].isna() | (users["city"] == "")
users.loc[null_cities, "city"] = "Unknown"

#Do the same for empty cities in events table
null_events_cities = events["city"].isna() | (events["city"] == "")
events.loc[null_events_cities,"city"] = "Unknown"

#Strip whitespace from user string and product columns
for name, i in [("users", users), ("products", products)]:
    str_cols = i.select_dtypes(include="object").columns
    for j in str_cols:
        i[j] = i[j].str.strip()


In [ ]:
#Exporting the cleaned data to file for feature generation in excel
users.to_csv("users_cleaned.csv",index=False)
products.to_csv("products_cleaned.csv",index=False)
orders.to_csv("orders_cleaned.csv",index=False)
events.to_csv("events_cleaned.csv",index=False)
order_items.to_csv("order_items_cleaned.csv",index=False)


In [ ]:
files.download("users_cleaned.csv")
files.download("products_cleaned.csv")
files.download("orders_cleaned.csv")
files.download("events_cleaned.csv")
files.download("order_items_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>